## Knowledge Distillation

Knowledge Distillation(KD, 지식 증류)는 고성능의 모델(Teacher)에서 지식을 전달 받아 상대적으로 간단한 모델(Student)을 학습시키는 방법

Teacher 모델이 내부 구조, 파라미터를 공개 여부에 따라 White-box, Black-box, Gray-box로 구분됩니다. Black-box는 결과만 확인 가능한 경우며  <br>
White-box는 모델의 내부 구조나 파라미터를 전부 알 수 있는 경우입니다. Gray-box는 그 중간으로 일부만 공개되어 있는 경우입니다. <br>
이러한 Teacher 모델의 특징에 의해 KD에 활용할 수 있는 정보의 종류가 달라지게 됩니다.


#### Reference:
https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html

https://github.com/NoCodeProgram/deepLearning/blob/main/transformer/KD_toy.ipynb

<br>

----
Pytorch Knowledge Distillation 튜토리얼을 기반으로 다음 내용을 학습습합니다.

(1) 모델 클래스 수정 방법

: 모델의 은닉 표현(hidden representation)을 추출하고, 이를 추가적인 계산에 활용할 수 있도록 모델 클래스를 수정하는 방법을 배웁니다.

(2) PyTorch 학습 루프 수정 방법

: 분류를 위한 Cross-Entropy 손실 외에, 추가적인 손실 함수를 포함하도록 PyTorch의 일반적인 학습 루프를 수정하는 방법을 배웁니다.

(3) 경량화 모델의 성능 향상 방법

보다 복잡하고 성능이 좋은 큰 모델(Teacher) 을 이용해, 작고 빠른 모델(Student) 의 성능을 향상시키는 방법을 배웁니다.

----

Packages import

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torch import Tensor

Torch Device

In [2]:
if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(f"Using device: {my_device}")

Using device: cuda


#### CIFAR-10 dataset

- 10개의 클래스
- 32x32 픽셀 이미지
- 10,000개의 이미지
- 50,000개의 트레이닝 이미지
- 10,000개의 테스트 이미지

입력 이미지는 RGB이므로 3개의 채널과 32x32 픽셀입니다. 기본적으로 각 이미지는 0에서 255까지의 3 x 32 x 32 = 3072개의 숫자로 표현됩니다. <br>
신경망에서 일반적인 관행은 입력을 정규화하는 것인데, 일반적으로 사용되는 활성화 함수에서 포화를 피하고 수치적 안정성을 높이는 것을 포함한 여러 가지 이유로 수행됩니다. <br>
정규화 프로세스는 각 채널의 평균을 빼고 표준 편차로 나누는 것으로 구성됩니다. 텐서 "mean=[0.485, 0.456, 0.406]"과 "std=[0.229, 0.224, 0.225]"는 이미 계산되었으며, <br>
이는 훈련 세트로 의도된 CIFAR-10의 사전 정의된 하위 세트에서 각 채널의 평균과 표준 편차를 나타냅니다. 평균과 표준 편차를 처음부터 다시 계산하지 않고 테스트 세트에도 이러한 값을 사용하는 방법에 주목하십시오. 이는 네트워크가 위의 숫자를 뺀 후 나누어 생성된 특징을 기반으로 학습되었기 때문이며, 일관성을 유지하고자 하기 때문입니다. 또한, 실제로는 테스트 세트의 평균과 표준 편차를 계산할 수 없습니다. 왜냐하면 우리의 가정에 따르면 해당 시점에는 테스트 세트에 접근할 수 없기 때문입니다.

  ![CIFAR-10](https://github.com/ultralytics/docs/releases/download/0/cifar10-sample-image.avif)

  Data References:
    1. https://www.cs.toronto.edu/~kriz/cifar.html <br>
    2. https://developer-together.tistory.com/49 <br>
    3. https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar

Training & testing data loading

In [5]:
import torch.utils


# batch size
batch_size = 128

# dataset for training
transform_train = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# dataset for validation
transform_test = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


#### Model class 

TeaterNet : Deeper CNN

StudentNet : Lightweight CNN

In [10]:
# Deeper CNN class to be used as TeacherNet
class TeacherNet(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.1):
        super().__init__()

        # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 32, 8, 8)
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),            
        )  

        # flatten step : (batch_size, 32, 8, 8) → .view(batch_size, 2048) 또는 .flatten(1)

        # classifier step : (batch_size, 2048) → (batch_size, num_classes)
        self.classifier = nn.Sequential(
           nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1) # = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
   



In [7]:
# Lightweight neural network class to be used as student:
class StudentNet(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.1):
        super().__init__()

        # feature extraction step : input (batch_size, 3, 32, 32) => output(batch_size, 16, 8, 8)
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
              
        # flatten step : (batch_size, 16, 8, 8) → .view(batch_size, 1024) 또는 .flatten(1)

        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

##### TeacherNet, StudentNet Parameters

각 모델의 파라미터 수를 출력하여 비교한다.

In [11]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

teacher_net = TeacherNet(num_classes=100)
student_net = StudentNet(num_classes=100)

print(f"TeacherNet has {count_parameters(teacher_net):,} trainable parameters")
print(f"StudentNet has {count_parameters(student_net):,} trainable parameters")

TeacherNet has 1,233,156 trainable parameters
StudentNet has 290,868 trainable parameters


#### TeacherNet vs StudentNet

TeacherNet 과 StudentNet 을 각각 학습하여 성능을 비교한다.

In [14]:
def train(model, train_loader, epochs, learning_rate, device):
  # loss function and optimizer
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)

  model.train()

  for epoch in range(epochs):
    running_loss = 0.0

    # inputs: A collection of batch_size images
    # labels: A vector of dimensionality batch_size with integers denoting class of each image
    for inputs, labels in train_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()

      # forward pass
      outputs = model(inputs)

      # outputs: Output of the network for the collection of images. A tensor of dimensionality batch_size x num_classes
      # labels: The actual labels of the images. Vector of dimensionality batch_size
      # criterion: The Cross-Entropy loss function
      loss = criterion(outputs, labels)
      loss.backward()

      # update weights
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


def test(model, test_loader, device):
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for inputs, labels in test_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      # forward pass
      outputs = model(inputs)

      _, predicted = torch.max(outputs.data, 1)
      
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Accuracy of the model on the test set: {accuracy:.2f}%")

  return accuracy


#### TeacherNet training

In [18]:
torch.manual_seed(42)

teacher_net = TeacherNet(num_classes=10).to(my_device)
train(teacher_net, train_loader, epochs=10, learning_rate=0.001, device=my_device)

# test accuracy
teacher_test_accuracy = test(teacher_net, test_loader, device=my_device)

Epoch 1/10, Loss: 1.3353117402557217
Epoch 2/10, Loss: 0.8622469115440193
Epoch 3/10, Loss: 0.6720652779959657
Epoch 4/10, Loss: 0.5316835739423552
Epoch 5/10, Loss: 0.40951780852911723
Epoch 6/10, Loss: 0.2994256889652413
Epoch 7/10, Loss: 0.21969498945471574
Epoch 8/10, Loss: 0.16420136623637147
Epoch 9/10, Loss: 0.14333714103168996
Epoch 10/10, Loss: 0.12007655674005713
Accuracy of the model on the test set: 75.51%


In [19]:
#### StudentNet instantiation
torch.manual_seed(42)
student_net = StudentNet(num_classes=10).to(my_device)

torch.manual_seed(42)
new_student_net = StudentNet(num_classes=10).to(my_device)

# 두 네트웍이 같은지 첫번째 레이어의 가중치 norm 을 출력
print("Norm of 1st layer of student_net:", torch.norm(student_net.features[0].weight).item())
print("Norm of 1st layer of new student_net:", torch.norm(new_student_net.features[0].weight).item())

Norm of 1st layer of student_net: 2.327361822128296
Norm of 1st layer of new student_net: 2.327361822128296


#### StudentNet training

In [20]:
train(student_net, train_loader, epochs=10, learning_rate=0.001, device=my_device)
student_test_accuracy = test(student_net, test_loader, device=my_device)

Epoch 1/10, Loss: 1.4691675111765752
Epoch 2/10, Loss: 1.1543239568505446
Epoch 3/10, Loss: 1.019650710818103
Epoch 4/10, Loss: 0.9172758331993962
Epoch 5/10, Loss: 0.8416361574016874
Epoch 6/10, Loss: 0.7763941734648117
Epoch 7/10, Loss: 0.7085736432038915
Epoch 8/10, Loss: 0.6524130961169368
Epoch 9/10, Loss: 0.5994498396621031
Epoch 10/10, Loss: 0.54791248431596
Accuracy of the model on the test set: 70.39%


##### Test Accuracy

In [21]:
print(f"TeacherNet accuracy: {teacher_test_accuracy:.2f}%")
print(f"StudentNet accuracy: {student_test_accuracy:.2f}%")

TeacherNet accuracy: 75.51%
StudentNet accuracy: 70.39%


## Knowledge Distillation (지식 증류)

#### First Method

지식 증류(knowledge distillation)는 두 네트워크 모두 클래스에 대해 확률 분포를 출력한다는 사실에 기반한 간단한 기법입니다. 

기존 Cross Entropy 손실에 교사 네트워크의 소프트맥스 출력을 기반으로 추가적인 손실로 통합함으로서 구현합니다. 

이는 적절하게 훈련된 교사 네트워크의 출력 활성화가 학습 중에 학생 네트워크가 활용할 수 있는 추가 정보를 전달한다는 가정에 기반합니다.

지식 증류에서는 **soft target** 의 전체 분포, 특히 작은 확률로 나타난 비정답 클래스 정보를 활용하면 모델이 학습 데이터의 내재된 구조와 클래스 간 관계를 더 잘 이해할 수 있고, 결과적으로 더 좋은 일반화 성능을 가지게 됩니다.


![First Method](https://docs.pytorch.org/tutorials/_static/img/knowledge_distillation/distillation_output_loss.png)

----

## Soft Targets Loss

##### (1) soft target 이란 ?

> **soft target** : 기존의 잘 학습된 큰 모델 (teacher)의 출력 <br>
<font color=skyblue>soft target = [고양이: 0.01, 개: 0.02, 자동차: 0.10, 트럭: 0.75, 비행기: 0.08, ...]</font> <br>
**hard target** : 보통 정답 하나만 있는 실제 label. 예: label = 3 (트럭)


##### (2) soft target 에서 작은 확률 값이 중요한가 ?

> 트럭(정답): 0.75, 자동차: 0.10, 비행기: 0.08, 고양이: 0.01 <br>
<font color=orange>**이 작은 값들이 알려주는 것은 트럭과 다른 클래스들 간의 유사성 정보입니다.**</font> <br>
자동차, 비행기 → 트럭과 시각적 유사성 있음 (교통수단, 바퀴 등) → 상대적으로 확률이 높음 <br>
고양이, 개 → 완전히 다른 클래스 → 확률이 거의 없음


"더 작은 확률" 값은 정답 클래스가 아니더라도, 그 확률이 얼마나 나왔는지를 통해 모델이 다른 클래스들과 얼마나 혼동하고 있는지를 알 수 있다는 것입니다. <br>
예로 모델이 트럭과 자동차를 혼동했다면 → 둘이 비슷한 특성이 있다는 뜻입니다. 이런 정보가 학습 데이터의 구조적 유사성을 잘 보존하는 데 기여합니다.


##### (3) soft target을 이용하면 왜 도움이 되나 ?

단순히 정답(hard target)만 학습하면 정답만 맞추고, 다른 클래스들과의 관계는 무시됩니다. 하지만 soft target 까지 학습하면 <br>

> <font color=orange>**"트럭 이라는 정답뿐만 아니라 자동차와 비행기도 좀 비슷하니까 헷갈릴 수 있어"**</font> 라는 클래스 간의 구조적 유사성을 모델이 배움으로서 더 일반화된 모델이 될 수 있습니다.


##### (4) Teacher vs Student 사이의 확률 분포 차이를 손실로 ?

KL Divergence를 이용하여 Teacher, Student 모델의 출력 값인 확률 분포의 차이 (soft targets loss)를 계산합니다. <br>
> <font color=green>**KL Divergence (Kullback–Leibler Divergence)는 확률 분포 간의 차이를 측정할 때 사용됩니다.**</font> <br>
> <br>
> KL(P∥Q)= ∑ P(i)log(P(i)/Q(i)) = ∑ P(i)(log(P(i)) - log(Q(i))) <br>
> <br>
𝑃 : 참(true) 확률 분포 (target) - Teacher Net의 결과 확률 분포 <br>
Q : 근사한 확률 분포 (prediction) - Student Net의 결과 확률 분포

➡ 이 식은 Q가 P를 얼마나 잘 설명하지 못하는지를 측정합니다. 즉, Q가 P와 다를수록 KL 값이 커집니다.

<br>

<font color=orange>**결과적으로 이 KL Divergence 를 기반으로 Loss로 정의하여 학습하면 Student는 Teacher를 잘 학습하게 됩니다.**</font>





In [24]:
""" 
  T : temperature : T 는 출력 분포의 부드러움을 제어합니다. 

  신경망의 마지막 출력(logits) 는 softmax 를 통해 클래스 확률로 바뀝니다.  
  
  softmax 는 가장 큰 값이 있는 위치에 거의 모든 확률을 몰아주는 경향이 있기 때문에 결과가 hard target 에 가까워집니다.
  
  지식 증류에서는 softmax 계산 시, 로짓을 온도 T로 나눠서 "더 부드러운" 분포를 만듭니다.

  T = 1	: 정답에 집중됨
  𝑇 > 1 : 분포가 평평해짐(부드러워짐)
  T < 1 : 분포가 더 날카롭게 변함 (확신 강함)
"""

#  teacher net >>> student net 으로 지식 증류
def train_knowledge_distillation(teacher_net, student_net, train_loader, epochs, learning_rate, 
                                 T, soft_loss_weight, hard_loss_weight, device):

  # loss function and optimizer for target student network
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(student_net.parameters(), lr=learning_rate)

  # teacher set to evaluation mode
  teacher_net.eval()

  # student set to training mode
  student_net.train()

  for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
      # Move data to device
      inputs, labels = inputs.to(device), labels.to(device)

      optimizer.zero_grad()

      # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
      with torch.no_grad():
        teacher_logits = teacher_net(inputs)      

      # forward pass
      student_logits = student_net(inputs)

      # total loss = weighted CrossEntropy(hard loss) + weighted KL_Divergence(soft loss)
      # Soften the teacher logits by applying softmax first
      teacher_soft_logits = nn.functional.softmax(teacher_logits / T, dim=1)

      # Soften the student logits by applying softmax first and log() second
      student_soft_logits = nn.functional.log_softmax(student_logits / T, dim=1)


      # Calculate the soft targets loss with KL divergence. 
      # Distillation 논문(Hinton et al., 2015)에서 온도 𝑇를 도입하면 loss 값이 작아져 Gradient Scale 도 작아짐
      # 따라서 최종 softloss 에서는  T**를 곱해 줌으로써 스케일을 복원합니다
      # soft_targets_loss = torch.sum(teacher_soft_logits * (teacher_soft_logits.log() - student_soft_logits)) / student_soft_logits.size()[0] * (T**2)
      soft_targets_loss = nn.functional.kl_div(student_soft_logits, teacher_soft_logits, reduction='batchmean') * (T ** 2)

      # Calculate the label loss (hard loss)
      hard_targets_loss = criterion(student_logits, labels)

      # Weighted sum of the two losses : CE + KD
      loss = soft_loss_weight * soft_targets_loss + hard_loss_weight * hard_targets_loss


      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


In [25]:
# Apply ``train_knowledge_distillation`` with a temperature of 2. Arbitrarily set the weights to 0.75 for CE and 0.25 for distillation loss.
train_knowledge_distillation(teacher_net=teacher_net, student_net=new_student_net, train_loader=train_loader, epochs=10, learning_rate=0.001, 
                             T=2, soft_loss_weight=0.25, hard_loss_weight=0.75, device=my_device)

kd_student_test_accuracy = test(new_student_net, test_loader, my_device)

# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {teacher_test_accuracy:.2f}%")
print(f"Student accuracy without teacher: {student_test_accuracy:.2f}%")
print(f"Student accuracy with CE + KD: {kd_student_test_accuracy:.2f}%")

Epoch 1/10, Loss: 2.409027507542954
Epoch 2/10, Loss: 1.8719159657388087
Epoch 3/10, Loss: 1.6480790989478227
Epoch 4/10, Loss: 1.4900165110292947
Epoch 5/10, Loss: 1.3611623390251413
Epoch 6/10, Loss: 1.2548418967315302
Epoch 7/10, Loss: 1.1563685257416552
Epoch 8/10, Loss: 1.0680859405976122
Epoch 9/10, Loss: 0.9863384861470489
Epoch 10/10, Loss: 0.9130013232950664
Accuracy of the model on the test set: 70.64%
Teacher accuracy: 75.51%
Student accuracy without teacher: 70.39%
Student accuracy with CE + KD: 70.64%
